Install Required Libraries

In [1]:
!pip install langchain langchain-community langchain-core
!pip install chromadb
!pip install pypdf
!pip install sentence-transformers
!pip install groq
!pip install langgraph
!pip install langchain-community
!pip install langchain-text-splitters

Import Libraries

In [2]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from groq import Groq

Set Groq API Key

In [3]:
os.environ["GROQ_API_KEY"] = "your API key"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

Upload PDF

In [4]:
from google.colab import files
uploaded = files.upload()

Saving sample.pdf to sample (1).pdf


Load the PDF

In [5]:
loader = PyPDFLoader("sample.pdf")
documents = loader.load()

print("Total pages:", len(documents))

Total pages: 3


Chunk the Document

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # size of each chunk
    chunk_overlap=50       # overlap for better context
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("Sample chunk:\n", chunks[0].page_content)

Total chunks: 12
Sample chunk:
 Frequently Asked Questions (FAQs)  
 
Q1. Can the Card be used immediately after it is purchased?  
Answer- Yes, your State Bank Vishwa Yatra  Foreign Travel Card can be used 
immediately after purchase except in India, Nepal and Bhutan.  
 
Q2. Can I use the Add-on cards simultaneously?  
Answer- Yes, Add-on cards can be used simultaneously along with the original card.  
 
Q3. Are there any expenses for which this Card may not be used?


Create Embeddings

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_10064/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Store in ChromaDB

In [8]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

print("Vector DB created successfully")

Vector DB created successfully


Create Retriever

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Test Retrieval

In [10]:
query = "Can I use the card for online transactions?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
prepaid.onlinesbi.com website and using your 16 digit card number and 6 digit 
password.  
 
Q13. Can I use the Card for on-line transactions?  
Answer: Presently, the Card is not enabled for e-Commerce. 
Q14. Do I need to have account with State Bank to purchase the Vishwa 
Yatra Foreign Travel Card? 
Answer: No, you need not be a State Bank account holder for purchasing State Bank 
Vishwa Yatra Foreign Travel Card. 
Q15. Can re-loading of the card by a family member or relatives possible?

--- Result 2 ---
prepaid.onlinesbi.com website and using your 16 digit card number and 6 digit 
password.  
 
Q13. Can I use the Card for on-line transactions?  
Answer: Presently, the Card is not enabled for e-Commerce. 
Q14. Do I need to have account with State Bank to purchase the Vishwa 
Yatra Foreign Travel Card? 
Answer: No, you need not be a State Bank account holder for purchasing State Bank 
Vishwa Yatra Foreign Travel Card. 
Q15. Can re-loading of the card by a family me

Create LLM Function

In [11]:
def generate_answer(query, context):
    prompt = f"""
You are a helpful customer support assistant.

Answer the question based ONLY on the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model="llama3-8b-8192",  # fast + free in Groq
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

Combine Retrieval + LLM

In [12]:
def rag_pipeline(query):
    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    answer = generate_answer(query, context)

    return answer

Test Chatbot

In [13]:
def generate_answer(query, context):
    prompt = f"""
You are a helpful customer support assistant.

Answer the question based ONLY on the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [14]:
!pip install langgraph

Define State

In [15]:
from typing import TypedDict

class GraphState(TypedDict):
    query: str
    context: str
    answer: str
    confidence: float

Processing Node

In [16]:
def processing_node(state):
    query = state["query"]

    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])

    answer = generate_answer(query, context)

    # Simple confidence logic
    if "I don't know" in answer:
        confidence = 0.2
    else:
        confidence = 0.9

    return {
        "context": context,
        "answer": answer,
        "confidence": confidence
    }

Decision Function

In [17]:
def route_decision(state):
    if state["confidence"] < 0.5:
        return "hitl"
    else:
        return "output"

Output Node

In [18]:
def output_node(state):
    print("\n✅ Final Answer:\n", state["answer"])
    return state

HITL Node

In [19]:
def hitl_node(state):
    print("\n⚠️ Escalating to Human Support...")
    print("Query:", state["query"])

    human_response = input("👨‍💻 Enter human response: ")

    return {
        "answer": human_response,
        "confidence": 1.0
    }

Build LangGraph

In [20]:
from langgraph.graph import StateGraph

builder = StateGraph(GraphState)

builder.add_node("process", processing_node)
builder.add_node("output", output_node)
builder.add_node("hitl", hitl_node)

builder.set_entry_point("process")

builder.add_conditional_edges(
    "process",
    route_decision,
    {
        "output": "output",
        "hitl": "hitl"
    }
)

builder.set_finish_point("output")

graph = builder.compile()

Run the Graph

In [22]:
query = "What is my account balance?"

graph.invoke({"query": query})


⚠️ Escalating to Human Support...
Query: What is my account balance?
👨‍💻 Enter human response: 45


{'query': 'What is my account balance?',
 'context': "made by account payee banker's cheque/ draft/ credit to account whereas if it is below \nRs. 50,000 Cash disbursement is also available.\n\nmade by account payee banker's cheque/ draft/ credit to account whereas if it is below \nRs. 50,000 Cash disbursement is also available.\n\nQ9. Do the ATM receipts show the amount withdrawn and the balance \navailable?  \nAnswer- Depending on the capability of the ATMs, the amount withdrawn and/or \nbalance available may be shown.  \n \nQ.10. In countries where ATM instructions are in an unfamiliar language, \nwhom can I approach for assistance?  \nAnswer- English is available at most VISA ATMs.  \n \nQ11. Is there a fee for obtaining refunds on the balance on the State Bank \nVishwa Yatra Foreign Travel Card?",
 'answer': '45',
 'confidence': 1.0}